In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F


In [17]:

##### MODEL ARCHITECTURE ######
class SelfAttentionzHead(nn.Module):
    def __init__(self,embedding_dim,block_size,head_size):
        super().__init__()
        self.query = nn.Linear(embedding_dim,head_size,bias=False)  
        self.key = nn.Linear(embedding_dim,head_size,bias=False)  
        self.value = nn.Linear(embedding_dim,head_size,bias=False) 
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))  
    def forward(self,x):
        B,T,C = x.shape  
        q = self.query(x)   
        k = self.query(x)  
        wei = q @ k.transpose(-2,-1) 
        wei = wei/(C ** 0.5)  
        wei = F.softmax(wei,dim=-1) 
        v = self.value(x) 
        out = wei @ v 
        return out


In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self,embedding_dim,block_size,num_heads):
        super().__init__()
        head_size = embedding_dim // num_heads  
        self.heads = ([SelfAttentionzHead(embedding_dim,block_size,head_size) for _ in range(num_heads)]) 
        self.proj = nn.Linear(num_heads * head_size,embedding_dim) 
    def forward(self,x):
        out = torch.concat([h(x) for h in self.heads],dim=-1)
        return self.proj(out)


In [19]:
class FNN(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(   
            nn.Linear(n_embd,4*n_embd),  
            nn.ReLU(),  
            nn.Linear(4*n_embd,n_embd)  
        )
    def forward(self,x):  
        out = self.net(x)
        return out	

In [20]:
class Block(nn.Module): 
    def __init__(self,embedding_dim,block_size,n_heads):
        super().__init__()
        self.sa = MultiHeadAttention(embedding_dim,block_size,n_heads)  
        self.ffn = FNN(embedding_dim)   
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)  
    def forward(self,x):   
        x = x + self.sa(self.ln1(x))  
        x = x + self.ffn(self.ln2(x)) 
        return x

In [21]:
####### MODEL TRAINING  ####### 
def PrepareData():
    corpus=["hello friends how are you",
    "the tea is very hot",
    "my name is Abhijeet",
    "the roads of delhi are busy",
    "it is raining in mumbai",
    "the train is late again",
    "i love eating samosas and drining tea",
    "holi is my favorite festival",
    "diwali brings lights and sweets",
    "india won the cricket match"
    ]
    new_corpus = ["<start> " + s + " <end>" for s in corpus]
    final_corpus = " ".join(new_corpus)
    
    chunks = final_corpus.split()
    vocab = list(set(chunks))
    vocab_size = len(vocab)
    embedding = {w:i for i,w in enumerate(vocab)}
    idx2word = {i: w for w, i in embedding.items()} 
    text_id_data = torch.tensor([embedding[c] for c in chunks],dtype=torch.long)
    return embedding,idx2word,text_id_data,vocab_size


In [22]:
def get_batch(batch_size=16):  
    index = torch.randint(len(text_id_data) - context_window,(batch_size,))     
    x = torch.stack([text_id_data[i:i+context_window] for i in index])
    y = torch.stack([text_id_data[i+1:i+context_window+1] for i in index])
    return x,y

In [23]:
class TinyGPT(nn.Module):  
    def __init__(self): 
        super().__init__()
        self.token_encoding = nn.Embedding(vocab_size,embedding_dim) 
        self.positional_encoding = nn.Embedding(context_window,embedding_dim)
        self.attentionblock = nn.Sequential(*[Block(embedding_dim,context_window,n_heads) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(embedding_dim) 
        self.head = nn.Linear(embedding_dim,vocab_size) 
        
    def forward(self,idx,target=None):
        B,T = idx.shape
        tok_enb = self.token_encoding(idx)
        pos_enb = self.positional_encoding(torch.arange(T))
        x = tok_enb + pos_enb
        x = self.attentionblock(x)
        x = self.ln(x)
        logits = self.head(x)
        loss = None
        
        if target is not None:
            B,T,C = logits.shape
            loss = F.cross_entropy(logits.view(B*T,C),target.view(B*T))
        return logits,loss
    
    def generate(self,idx,max_new_token):  
        for _ in range(max_new_token):  
             idx_cond = idx[:,-context_window:] 
             logists,_ = self(idx_cond)   
             logists = logists[:,-1,:]  
             prob = F.softmax(logists,dim=-1) 
             new_idx = torch.multinomial(prob,1)  
             idx = torch.concat((idx,new_idx),dim=1) 
        return idx



In [26]:
def TrainModel():
    model = TinyGPT()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lerning_rate)
    for step in range(epoch):
            xb, yb = get_batch() 
            logits, loss = model(xb, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if step % 300 == 0:
                print(f"Step {step}, loss={loss.item():.4f}")
    return model,logits,loss

In [27]:
if __name__ == '__main__':
    context_window = 6  
    embedding_dim = 32  
    n_heads = 2  
    n_layers = 2  
    lerning_rate = 0.01 
    epoch = 1500
    embedding,idx2word,text_id_data,vocab_size = PrepareData()
    model,logits,loss = TrainModel()    

    context = torch.tensor([[embedding["hello"]]], dtype=torch.long)
    out = model.generate(context, max_new_token=15)

    print("\nGenerated text:\n")
    print(" ".join(idx2word[int(i)] for i in out[0]))

Step 0, loss=3.9165
Step 300, loss=0.0014
Step 600, loss=0.0006
Step 900, loss=0.0004
Step 1200, loss=0.0002

Generated text:

hello friends how are you <end> <start> the tea is very hot <end> <start> my name
